# 02 — Connectome reservoir on Iris

Same Iris benchmark as notebook 01, but the recurrent matrix is a **brain connectome** (`reservoirs.connectome.ConnectomeReservoir`) instead of a random matrix — so you can compare biological vs random connectivity on identical data. Runs on the committed **mock** 60-node connectome; point `GRAPH_DIR` at a real connectome folder for paper-grade results.

> **Limitation:** structural connectomes from non-invasive imaging are *undirected*, so `symmetric=True` is correct here and the reservoir has a real-eigenvalue-only spectrum (no oscillatory modes).

In [1]:
import sys, os
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'reservoirs').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO)); os.chdir(REPO)
print('repo root set:', REPO.name)   # name only, no absolute path

repo root set: ESN-unofficial-1-utilizzando-matrici-random-con-iris


In [2]:
import numpy as np, pandas as pd, random
df = pd.get_dummies(pd.read_csv('iris.csv', index_col=0))
data_raw = df.to_numpy().astype(float)
rng = random.Random(7)
sample = lambda a, b: rng.sample(range(a, b), 40)
train_idxs = sample(0,50)+sample(50,100)+sample(100,150)
test_idxs = [i for i in range(150) if i not in train_idxs]
data = data_raw.copy()
data[:, :4] = data[:, :4] / data[train_idxs, :4].max(axis=0)
timesteps = np.arange(0, 50, 0.5)   # 100 steps (raise resolution -> higher accuracy, slower)
u = np.array([np.vstack([np.sin(timesteps*2*np.pi*pt[i]) for i in range(4)]).T for pt in data[:, :4]])
y = np.array([data[:, 4:]] * len(timesteps)).swapaxes(0, 1).astype(float)
u_train, u_test, y_train, y_test = u[train_idxs], u[test_idxs], y[train_idxs], y[test_idxs]
print('iris encoded:', u.shape, '-> classes', y.shape[-1])

iris encoded: (150, 100, 4) -> classes 3


In [3]:
WASHOUT, ALPHA = 10, 1e-4
def collect(res, batch):
    return np.stack([res.forward(seq, collect_states=True) for seq in batch], axis=0)
def ridge(Xs, Ys, washout=WASHOUT, alpha=ALPHA):
    X = Xs[:, washout:, :].reshape(-1, Xs.shape[-1]); Y = Ys[:, washout:, :].reshape(-1, Ys.shape[-1])
    return np.linalg.solve(X.T @ X + alpha*np.eye(X.shape[1]), X.T @ Y)
def series_acc(pred, Yt):
    return float(np.mean(pred.mean(1).argmax(1) == Yt.mean(1).argmax(1)))
def evaluate(res):
    Xtr, Xte = collect(res, u_train), collect(res, u_test)
    w = ridge(Xtr, y_train)
    return series_acc(Xtr @ w, y_train), series_acc(Xte @ w, y_test)

In [4]:
import warnings
from reservoirs.connectome import ConnectomeReservoir
GRAPH_DIR = 'generated_artifacts/graphs'   # <- swap for a real connectome folder
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    res = ConnectomeReservoir(4, graph_dir=GRAPH_DIR, spectral_radius=0.9, seed=7)
tr, te = evaluate(res)
print(f'connectome reservoir  N={res.n_neurons}  train_acc={tr:.3f}  test_acc={te:.3f}')

connectome reservoir  N=60  train_acc=0.850  test_acc=0.733


Compared with notebook 01's random reservoirs of the same size, this shows whether the empirical connectome topology helps on Iris. A proper claim also needs the **null-model baselines** (degree-preserving rewired + size-matched random ESN) — added in Phase 4.